## Bronze Layer Ingestion

**Pipeline Flow**

Raw CSV files  
↓  
Read with Spark  
↓  
Add ingestion metadata  
↓  
Write as Delta  
↓  
**Bronze tables**

### 1. Configuration

Define source paths, input files, and target Bronze layer configuration.

In [0]:
# Source configuration
base_path = "/Workspace/Users/volkan.kulturoglu@windowslive.com/DbAcademy/Data/"

source_files = {
    "transactions": "transaction_data.csv",
    "products": "product.csv",
    "demographics": "hh_demographic.csv",
    "campaign_desc": "campaign_desc.csv",
    "campaign_households": "campaign_table.csv",
    "coupons": "coupon.csv",
    "coupon_redemptions": "coupon_redempt.csv"
}

# Target configuration
bronze_catalog = "workspace"
bronze_schema = "consumer_analytics_bronze"

### 2. Create Bronze Schema

Create a dedicated schema for Bronze Delta tables.

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS workspace.consumer_analytics_bronze
""")

DataFrame[]

In [0]:
spark.sql("SHOW SCHEMAS IN workspace").show(truncate=False)

+-------------------------+
|databaseName             |
+-------------------------+
|consumer_analytics_bronze|
|default                  |
|information_schema       |
+-------------------------+



### 3. Read Source CSV Files

Load all source CSV files into Spark DataFrames using reusable ingestion logic.

In [0]:
bronze_dfs = {}

for table_name, file_name in source_files.items():

    file_path = base_path + file_name

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(file_path)
    )

    bronze_dfs[table_name] = df

    print(f"{table_name}: loaded")

transactions: loaded
products: loaded
demographics: loaded
campaign_desc: loaded
campaign_households: loaded
coupons: loaded
coupon_redemptions: loaded


### 4. Add Ingestion Metadata

Add technical metadata columns to improve data lineage and traceability within the Bronze layer.

- `_source_file`: identifies the source file of each record.
- `_ingested_at`: records when the data was ingested into the pipeline.

In [0]:
from pyspark.sql import functions as F

for table_name, file_name in source_files.items():

    bronze_dfs[table_name] = (
        bronze_dfs[table_name]
        .withColumn("_source_file", F.lit(file_name))
        .withColumn("_ingested_at", F.current_timestamp())
    )

print("Ingestion metadata added to all datasets.")

Ingestion metadata added to all datasets.


### 5. Validate Ingestion Metadata

Validate that the technical metadata columns were successfully added before persisting the Bronze tables.

In [0]:
display(
    bronze_dfs["transactions"]
    .select(
        "household_key",
        "BASKET_ID",
        "PRODUCT_ID",
        "SALES_VALUE",
        "_source_file",
        "_ingested_at"
    )
    .limit(10)
)

household_key,BASKET_ID,PRODUCT_ID,SALES_VALUE,_source_file,_ingested_at
2375,26984851472,1004906,1.39,transaction_data.csv,2026-09-23T09:07:41.466Z
2375,26984851472,1033142,0.82,transaction_data.csv,2026-09-23T09:07:41.466Z
2375,26984851472,1036325,0.99,transaction_data.csv,2026-09-23T09:07:41.466Z
2375,26984851472,1082185,1.21,transaction_data.csv,2026-09-23T09:07:41.466Z
2375,26984851472,8160430,1.5,transaction_data.csv,2026-09-23T09:07:41.466Z
2375,26984851516,826249,1.98,transaction_data.csv,2026-09-23T09:07:41.466Z
2375,26984851516,1043142,1.57,transaction_data.csv,2026-09-23T09:07:41.466Z
2375,26984851516,1085983,2.99,transaction_data.csv,2026-09-23T09:07:41.466Z
2375,26984851516,1102651,1.89,transaction_data.csv,2026-09-23T09:07:41.466Z
2375,26984851516,6423775,2.0,transaction_data.csv,2026-09-23T09:07:41.466Z


### 6. Write Bronze Delta Tables

Persist the source-aligned DataFrames as managed Delta tables in the Bronze schema.

The Bronze layer preserves the original source data while adding technical ingestion metadata for lineage and traceability.

In [0]:
for table_name, df in bronze_dfs.items():

    target_table = f"{bronze_catalog}.{bronze_schema}.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    print(f"{target_table}: written successfully")

workspace.consumer_analytics_bronze.transactions: written successfully
workspace.consumer_analytics_bronze.products: written successfully
workspace.consumer_analytics_bronze.demographics: written successfully
workspace.consumer_analytics_bronze.campaign_desc: written successfully
workspace.consumer_analytics_bronze.campaign_households: written successfully
workspace.consumer_analytics_bronze.coupons: written successfully
workspace.consumer_analytics_bronze.coupon_redemptions: written successfully


### 7. Validate Bronze Tables

Verify that all expected Bronze tables were created successfully.

In [0]:
display(
    spark.sql(
        f"SHOW TABLES IN {bronze_catalog}.{bronze_schema}"
    )
)

database,tableName,isTemporary
consumer_analytics_bronze,campaign_desc,false
consumer_analytics_bronze,campaign_households,false
consumer_analytics_bronze,coupon_redemptions,false
consumer_analytics_bronze,coupons,false
consumer_analytics_bronze,demographics,false
consumer_analytics_bronze,products,false
consumer_analytics_bronze,transactions,false


### 8. Validate Source-to-Bronze Row Counts

Compare source DataFrame row counts with Bronze Delta table row counts to confirm that ingestion preserved all source records.

In [0]:
for table_name, source_df in bronze_dfs.items():

    target_table = f"{bronze_catalog}.{bronze_schema}.{table_name}"

    source_count = source_df.count()
    bronze_count = spark.table(target_table).count()

    status = "PASS" if source_count == bronze_count else "FAIL"

    print(
        f"{table_name}: "
        f"source={source_count:,} | "
        f"bronze={bronze_count:,} | "
        f"{status}"
    )

transactions: source=2,595,732 | bronze=2,595,732 | PASS
products: source=92,353 | bronze=92,353 | PASS
demographics: source=801 | bronze=801 | PASS
campaign_desc: source=30 | bronze=30 | PASS
campaign_households: source=7,208 | bronze=7,208 | PASS
coupons: source=124,548 | bronze=124,548 | PASS
coupon_redemptions: source=2,318 | bronze=2,318 | PASS
